# Spider 1.0: официальный test

Ноутбук повторяет все пять замеров участника №2 на 2147 вопросах из 40 баз. Используйте T4 GPU. Загрузите spider_rerun_adapters.zip в /content через панель Files. Архив содержит два готовых LoRA-адаптера. Исходный Spider скачивается из официального источника или его зеркала с проверкой SHA-256.

В /content/spider-test-results сохраняются контрольные точки по 128 вопросам и отдельный ZIP каждого завершённого варианта. Отдельный ZIP со всеми результатами создаётся в конце. При потере отображения вывода проверяйте журнал /content/spider-test-rerun.log и папку результатов.

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
print(torch.cuda.get_device_name(0), torch.__version__)
ROOT = Path('/content/spider-official-rerun')
OUT = Path('/content/spider-test-results')
ARCHIVE = Path('/content/spider_data.zip')
ADAPTERS = Path('/content/spider_rerun_adapters.zip')
BRANCH = 'participant-2/evaluation'
REPO = 'https://github.com/Pavel-V418/BaseModel-vs-SFT.git'
ARCHIVE_SHA = '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def valid_archive(path):
    if not path.is_file():
        return False
    with path.open('rb') as source:
        return hashlib.file_digest(source, 'sha256').hexdigest() == ARCHIVE_SHA

if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(ROOT)], check=True)
SOURCE = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements-colab.txt')], check=True)
if not valid_archive(ARCHIVE):
    temp = ARCHIVE.with_suffix('.download')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    try:
        subprocess.run([sys.executable, '-m', 'gdown', '1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J',
                        '-O', str(temp)], check=True, timeout=90)
    except (subprocess.SubprocessError, OSError) as error:
        print('Official download unavailable, trying verified mirror:', error, flush=True)
    if not valid_archive(temp):
        url = 'https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip'
        subprocess.run(['curl','-fL','--retry','2','--max-time','600','-o',str(temp),url],check=True)
    assert valid_archive(temp), 'Spider archive checksum differs'
    temp.replace(ARCHIVE)

In [ ]:
cmd = [sys.executable, '-u', '-m', 'src.spider_test', '--archive', str(ARCHIVE),
       '--data-dir', '/content/spider-test-data', '--adapters', str(ADAPTERS),
       '--results', str(OUT), '--source-commit', SOURCE]
with Path('/content/spider-test-rerun.log').open('a', encoding='utf-8') as log:
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8') as run:
        for line in run.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    if run.returncode:
        raise subprocess.CalledProcessError(run.returncode, cmd)

In [ ]:
files_to_export = sorted(OUT.glob('*_eval.json*')) + sorted(OUT.glob('*_eval_metrics.json'))
assert len(set(files_to_export)) == 10
target = '/content/spider_official_test_results.zip'
with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(files_to_export)):
        archive.write(path, path.name)
from google.colab import files
files.download(target)